In [1]:
import cv2

In [3]:
import cv2
import numpy as np
import os


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    # Convert to grayscale if needed
    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Bilateral filtering
    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    # Canny edge detection
    edges = cv2.Canny(blur, 50, 120)

    # Morphological closing
    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# CONTOUR DRAWING
# =====================================================

def draw_large_contours(frame, edges, min_area=9000):

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    output = frame.copy()

    kept_count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            kept_count += 1

    return output, kept_count


# =====================================================
# FRAME ENHANCEMENT PIPELINE
# =====================================================

def enhance_frame(frame):

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Denoising
    denoised = gaussian_denoise(gray)

    # Contrast stretching
    stretched = contrast_stretch(denoised)

    # Sharpening
    sharpened = sharpen_image(stretched)

    # Convert back to BGR
    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    # Open input video
    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    # Read video properties
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    if not writer.isOpened():
        print("Error creating output video.")
        return

    frames_processed = 0
    frames_with_segments = 0

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    # =================================================
    # FRAME PROCESSING LOOP
    # =================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # Enhancement pipeline
        enhanced = enhance_frame(frame)

        # Edge detection pipeline
        edges = build_edges(enhanced)

        # Contour segmentation
        output_frame, kept_count = draw_large_contours(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        # Count segmented frames
        if kept_count > 0:
            frames_with_segments += 1

        # Save frame
        writer.write(output_frame)

        # Preview window
        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        frames_processed += 1

        # Press q to quit
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    # =================================================
    # RELEASE RESOURCES
    # =================================================

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("\nPipeline completed.")
    print("Frames processed:", frames_processed)
    print("Frames with segmentation:", frames_with_segments)
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.

Pipeline completed.
Frames processed: 1004
Frames with segmentation: 1001
Output saved as: IMG_7555_segmented.mp4


In [4]:
import cv2
import numpy as np
import os


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    # Convert to grayscale if needed
    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Bilateral filtering
    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    # Canny edge detection
    edges = cv2.Canny(blur, 50, 120)

    # Morphological closing
    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# CONTOUR DRAWING
# =====================================================

def draw_large_contours(frame, edges, min_area=9000):

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    output = frame.copy()

    kept_count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            kept_count += 1

    return output, kept_count


# =====================================================
# FRAME ENHANCEMENT PIPELINE
# =====================================================

def enhance_frame(frame):

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Denoising
    denoised = gaussian_denoise(gray)

    # Contrast stretching
    stretched = contrast_stretch(denoised)

    # Sharpening
    sharpened = sharpen_image(stretched)

    # Convert back to BGR
    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    # Open input video
    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    # Read video properties
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    if not writer.isOpened():
        print("Error creating output video.")
        return

    frames_processed = 0
    frames_with_segments = 0

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    # =================================================
    # FRAME PROCESSING LOOP
    # =================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # Enhancement pipeline
        enhanced = enhance_frame(frame)

        # Edge detection pipeline
        edges = build_edges(enhanced)

        # Contour segmentation
        output_frame, kept_count = draw_large_contours(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        # =================================================
        # ADD ROI BOXES
        # =================================================

        # Get frame dimensions
        h, w = output_frame.shape[:2]

        # ROI dimensions
        roi_width = int(w * 0.22)
        roi_height = int(h * 0.18)

        # Bottom margin
        bottom_margin = 20

        # LEFT ROI (Light Green)
        left_x1 = int(w * 0.08)
        left_y1 = h - roi_height - bottom_margin
        left_x2 = left_x1 + roi_width
        left_y2 = h - bottom_margin

        cv2.rectangle(
            output_frame,
            (left_x1, left_y1),
            (left_x2, left_y2),
            (144, 238, 144),   # Light Green
            3
        )

        # CENTER ROI (Light Blue)
        center_x1 = (w // 2) - (roi_width // 2)
        center_y1 = h - roi_height - bottom_margin
        center_x2 = center_x1 + roi_width
        center_y2 = h - bottom_margin

        cv2.rectangle(
            output_frame,
            (center_x1, center_y1),
            (center_x2, center_y2),
            (255, 255, 0),     # Light Blue
            3
        )

        # RIGHT ROI (Light Green)
        right_x2 = int(w * 0.92)
        right_x1 = right_x2 - roi_width
        right_y1 = h - roi_height - bottom_margin
        right_y2 = h - bottom_margin

        cv2.rectangle(
            output_frame,
            (right_x1, right_y1),
            (right_x2, right_y2),
            (144, 238, 144),   # Light Green
            3
        )

        # Count segmented frames
        if kept_count > 0:
            frames_with_segments += 1

        # Save frame
        writer.write(output_frame)

        # Preview window
        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        frames_processed += 1

        # Press q to quit
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    # =================================================
    # RELEASE RESOURCES
    # =================================================

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("\nPipeline completed.")
    print("Frames processed:", frames_processed)
    print("Frames with segmentation:", frames_with_segments)
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.

Pipeline completed.
Frames processed: 72
Frames with segmentation: 71
Output saved as: IMG_7555_segmented.mp4


In [5]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    # Convert to grayscale
    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Bilateral filtering
    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    # Canny edge detection
    edges = cv2.Canny(blur, 50, 120)

    # Morphological closing
    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# ROI-BASED CONTOUR DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.24)
    side_roi_height = int(h * 0.32)

    center_roi_width = int(w * 0.52)
    center_roi_height = int(h * 0.40)

    bottom_margin = 10

    # =================================================
    # LEFT ROI
    # =================================================

    left_x1 = int(w * 0.015)
    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)
    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    right_x2 = int(w * 0.985)
    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASK
    # =================================================

    roi_mask = np.zeros(edges.shape, dtype=np.uint8)

    # Left ROI
    cv2.rectangle(
        roi_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    # Center ROI
    cv2.rectangle(
        roi_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    # Right ROI
    cv2.rectangle(
        roi_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # Apply ROI mask
    roi_edges = cv2.bitwise_and(edges, roi_mask)

    # =================================================
    # FIND CONTOURS ONLY INSIDE ROI
    # =================================================

    contours, _ = cv2.findContours(
        roi_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    kept_count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            kept_count += 1

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    # LEFT ROI
    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    # CENTER ROI
    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    # RIGHT ROI
    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output, kept_count


# =====================================================
# FRAME ENHANCEMENT PIPELINE
# =====================================================

def enhance_frame(frame):

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Denoising
    denoised = gaussian_denoise(gray)

    # Contrast stretching
    stretched = contrast_stretch(denoised)

    # Sharpening
    sharpened = sharpen_image(stretched)

    # Convert back to BGR
    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    if not writer.isOpened():
        print("Error creating output video.")
        return

    frames_processed = 0
    frames_with_segments = 0

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    # =================================================
    # FRAME LOOP
    # =================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # Enhancement pipeline
        enhanced = enhance_frame(frame)

        # Edge detection
        edges = build_edges(enhanced)

        # ROI-based segmentation
        output_frame, kept_count = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        # Count segmented frames
        if kept_count > 0:
            frames_with_segments += 1

        # Save frame
        writer.write(output_frame)

        # Preview
        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        frames_processed += 1

        # Quit with q
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    # =================================================
    # RELEASE
    # =================================================

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("\nPipeline completed.")
    print("Frames processed:", frames_processed)
    print("Frames with segmentation:", frames_with_segments)
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.

Pipeline completed.
Frames processed: 163
Frames with segmentation: 161
Output saved as: IMG_7555_segmented.mp4


In [6]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    # Convert to grayscale
    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Bilateral filtering
    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    # Canny edge detection
    edges = cv2.Canny(blur, 50, 120)

    # Morphological closing
    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# ROI-BASED CONTOUR DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.26)

    # 2x taller side ROIs
    side_roi_height = int(h * 0.62)

    # Much wider center ROI
    center_roi_width = int(w * 0.72)

    # Much taller center ROI
    center_roi_height = int(h * 0.75)

    # Bottom spacing
    bottom_margin = 5

    # =================================================
    # LEFT ROI
    # =================================================

    # Touch left edge
    left_x1 = 0

    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)

    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    # Touch right edge
    right_x2 = w

    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASK
    # =================================================

    roi_mask = np.zeros(edges.shape, dtype=np.uint8)

    # Left ROI
    cv2.rectangle(
        roi_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    # Center ROI
    cv2.rectangle(
        roi_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    # Right ROI
    cv2.rectangle(
        roi_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # Apply ROI mask
    roi_edges = cv2.bitwise_and(edges, roi_mask)

    # =================================================
    # FIND CONTOURS INSIDE ROI ONLY
    # =================================================

    contours, _ = cv2.findContours(
        roi_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    kept_count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            kept_count += 1

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    # LEFT ROI
    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    # CENTER ROI
    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    # RIGHT ROI
    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output, kept_count


# =====================================================
# FRAME ENHANCEMENT PIPELINE
# =====================================================

def enhance_frame(frame):

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Denoising
    denoised = gaussian_denoise(gray)

    # Contrast stretching
    stretched = contrast_stretch(denoised)

    # Sharpening
    sharpened = sharpen_image(stretched)

    # Convert back to BGR
    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    if not writer.isOpened():
        print("Error creating output video.")
        return

    frames_processed = 0
    frames_with_segments = 0

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    # =================================================
    # FRAME LOOP
    # =================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # Enhancement pipeline
        enhanced = enhance_frame(frame)

        # Edge detection
        edges = build_edges(enhanced)

        # ROI segmentation
        output_frame, kept_count = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        # Count segmented frames
        if kept_count > 0:
            frames_with_segments += 1

        # Save frame
        writer.write(output_frame)

        # Preview
        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        frames_processed += 1

        # Press q to quit
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    # =================================================
    # RELEASE
    # =================================================

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("\nPipeline completed.")
    print("Frames processed:", frames_processed)
    print("Frames with segmentation:", frames_with_segments)
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()
    

Starting segmentation pipeline...
Press q to stop preview.

Pipeline completed.
Frames processed: 405
Frames with segmentation: 403
Output saved as: IMG_7555_segmented.mp4


In [8]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    # Convert to grayscale
    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Bilateral filtering
    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    # Canny edge detection
    edges = cv2.Canny(blur, 50, 120)

    # Morphological closing
    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# ROI-BASED CONTOUR DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

  # =================================================
# ROI SETTINGS
# =================================================

# LEFT + RIGHT ROI
side_roi_width = int(w * 0.18)
side_roi_height = int(h * 0.66)

# CENTER ROI (VERY LARGE)
center_roi_width = int(w * 0.70)
center_roi_height = int(h * 0.78)

bottom_margin = 5

# =================================================
# LEFT ROI
# =================================================

left_x1 = 0
left_y1 = h - side_roi_height - bottom_margin

left_x2 = left_x1 + side_roi_width
left_y2 = h - bottom_margin

# =================================================
# CENTER ROI
# =================================================

center_x1 = (w // 2) - (center_roi_width // 2)
center_y1 = h - center_roi_height - bottom_margin

center_x2 = center_x1 + center_roi_width
center_y2 = h - bottom_margin

# =================================================
# RIGHT ROI
# =================================================

right_x2 = w
right_x1 = right_x2 - side_roi_width

right_y1 = h - side_roi_height - bottom_margin
right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASK
    # =================================================

    roi_mask = np.zeros(edges.shape, dtype=np.uint8)

    # Left ROI
    cv2.rectangle(
        roi_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    # Center ROI
    cv2.rectangle(
        roi_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    # Right ROI
    cv2.rectangle(
        roi_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # Apply ROI mask
    roi_edges = cv2.bitwise_and(edges, roi_mask)

    # =================================================
    # FIND CONTOURS INSIDE ROI ONLY
    # =================================================

    contours, _ = cv2.findContours(
        roi_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    kept_count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            kept_count += 1

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    # LEFT ROI
    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    # CENTER ROI
    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    # RIGHT ROI
    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output, kept_count


# =====================================================
# FRAME ENHANCEMENT PIPELINE
# =====================================================

def enhance_frame(frame):

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Denoising
    denoised = gaussian_denoise(gray)

    # Contrast stretching
    stretched = contrast_stretch(denoised)

    # Sharpening
    sharpened = sharpen_image(stretched)

    # Convert back to BGR
    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    if not writer.isOpened():
        print("Error creating output video.")
        return

    frames_processed = 0
    frames_with_segments = 0

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    # =================================================
    # FRAME LOOP
    # =================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # Enhancement pipeline
        enhanced = enhance_frame(frame)

        # Edge detection
        edges = build_edges(enhanced)

        # ROI segmentation
        output_frame, kept_count = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        # Count segmented frames
        if kept_count > 0:
            frames_with_segments += 1

        # Save frame
        writer.write(output_frame)

        # Preview
        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        frames_processed += 1

        # Press q to quit
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    # =================================================
    # RELEASE
    # =================================================

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("\nPipeline completed.")
    print("Frames processed:", frames_processed)
    print("Frames with segmentation:", frames_with_segments)
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

IndentationError: unexpected indent (940214984.py, line 157)

In [12]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    process_video()

In [13]:
# ROI-Based Road Damage Detection

python
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    edges = cv2.Canny(blur, 50, 120)

    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# FRAME ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    denoised = gaussian_denoise(gray)

    stretched = contrast_stretch(denoised)

    sharpened = sharpen_image(stretched)

    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# ROI DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.18)
    side_roi_height = int(h * 0.66)

    center_roi_width = int(w * 0.70)
    center_roi_height = int(h * 0.78)

    bottom_margin = 5

    # =================================================
    # LEFT ROI
    # =================================================

    left_x1 = 0
    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)
    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    right_x2 = w
    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASKS
    # =================================================

    left_mask = np.zeros(edges.shape, dtype=np.uint8)
    center_mask = np.zeros(edges.shape, dtype=np.uint8)
    right_mask = np.zeros(edges.shape, dtype=np.uint8)

    cv2.rectangle(
        left_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    cv2.rectangle(
        center_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    cv2.rectangle(
        right_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # =================================================
    # APPLY ROI MASKS
    # =================================================

    left_edges = cv2.bitwise_and(edges, left_mask)
    center_edges = cv2.bitwise_and(edges, center_mask)
    right_edges = cv2.bitwise_and(edges, right_mask)

    # =================================================
    # LEFT ROI -> EDGE CRACKS
    # =================================================

    contours_left, _ = cv2.findContours(
        left_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_left:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

    # Label
    cv2.putText(
        output,
        "EDGE CRACKS",
        (left_x1 + 10, left_y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (144, 238, 144),
        2
    )

    # =================================================
    # CENTER ROI -> POTHOLES + RAVELLING
    # =================================================

    contours_center, _ = cv2.findContours(
        center_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_center:

        area = cv2.contourArea(cnt)

        if area > min_area:

            x, y, ww, hh = cv2.boundingRect(cnt)

            # Pothole condition
            if area > 25000:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (0, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "POTHOLE",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2
                )

            # Ravelling condition
            else:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (255, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "RAVELLING",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 0, 255),
                    2
                )

    # =================================================
    # RIGHT ROI -> EDGE CRACKS
    # =================================================

    contours_right, _ = cv2.findContours(
        right_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_right:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

    # Label
    cv2.putText(
        output,
        "EDGE CRACKS",
        (right_x1 + 10, right_y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (144, 238, 144),
        2
    )

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        enhanced = enhance_frame(frame)

        edges = build_edges(enhanced)

        output_frame = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        writer.write(output_frame)

        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("Pipeline completed.")
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()



NameError: name 'python' is not defined

In [14]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    edges = cv2.Canny(blur, 50, 120)

    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# FRAME ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    denoised = gaussian_denoise(gray)

    stretched = contrast_stretch(denoised)

    sharpened = sharpen_image(stretched)

    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# ROI DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.18)
    side_roi_height = int(h * 0.66)

    center_roi_width = int(w * 0.70)
    center_roi_height = int(h * 0.78)

    bottom_margin = 5

    # =================================================
    # LEFT ROI
    # =================================================

    left_x1 = 0
    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)
    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    right_x2 = w
    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASKS
    # =================================================

    left_mask = np.zeros(edges.shape, dtype=np.uint8)
    center_mask = np.zeros(edges.shape, dtype=np.uint8)
    right_mask = np.zeros(edges.shape, dtype=np.uint8)

    cv2.rectangle(
        left_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    cv2.rectangle(
        center_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    cv2.rectangle(
        right_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # =================================================
    # APPLY ROI MASKS
    # =================================================

    left_edges = cv2.bitwise_and(edges, left_mask)
    center_edges = cv2.bitwise_and(edges, center_mask)
    right_edges = cv2.bitwise_and(edges, right_mask)

    # =================================================
    # LEFT ROI -> EDGE CRACKS
    # =================================================

    contours_left, _ = cv2.findContours(
        left_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_left:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

    cv2.putText(
        output,
        "EDGE CRACKS",
        (left_x1 + 10, left_y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (144, 238, 144),
        2
    )

    # =================================================
    # CENTER ROI -> POTHOLES + RAVELLING
    # =================================================

    contours_center, _ = cv2.findContours(
        center_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_center:

        area = cv2.contourArea(cnt)

        if area > min_area:

            x, y, ww, hh = cv2.boundingRect(cnt)

            # POTHOLES
            if area > 25000:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (0, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "POTHOLE",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2
                )

            # RAVELLING
            else:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (255, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "RAVELLING",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 0, 255),
                    2
                )

    # =================================================
    # RIGHT ROI -> EDGE CRACKS
    # =================================================

    contours_right, _ = cv2.findContours(
        right_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_right:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

    cv2.putText(
        output,
        "EDGE CRACKS",
        (right_x1 + 10, right_y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (144, 238, 144),
        2
    )

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        enhanced = enhance_frame(frame)

        edges = build_edges(enhanced)

        output_frame = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        writer.write(output_frame)

        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("Pipeline completed.")
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.
Pipeline completed.
Output saved as: IMG_7555_segmented.mp4


In [15]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    edges = cv2.Canny(blur, 50, 120)

    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# FRAME ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    denoised = gaussian_denoise(gray)

    stretched = contrast_stretch(denoised)

    sharpened = sharpen_image(stretched)

    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# ROI DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.18)
    side_roi_height = int(h * 0.66)

    center_roi_width = int(w * 0.70)
    center_roi_height = int(h * 0.78)

    bottom_margin = 5

    # =================================================
    # LEFT ROI
    # =================================================

    left_x1 = 0
    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)
    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    right_x2 = w
    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASKS
    # =================================================

    left_mask = np.zeros(edges.shape, dtype=np.uint8)
    center_mask = np.zeros(edges.shape, dtype=np.uint8)
    right_mask = np.zeros(edges.shape, dtype=np.uint8)

    cv2.rectangle(
        left_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    cv2.rectangle(
        center_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    cv2.rectangle(
        right_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # =================================================
    # APPLY ROI MASKS
    # =================================================

    left_edges = cv2.bitwise_and(edges, left_mask)
    center_edges = cv2.bitwise_and(edges, center_mask)
    right_edges = cv2.bitwise_and(edges, right_mask)

    # =================================================
    # LEFT ROI -> EDGE CRACKS
    # =================================================

    contours_left, _ = cv2.findContours(
        left_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_left:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

    cv2.putText(
        output,
        "EDGE CRACKS",
        (left_x1 + 10, left_y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (144, 238, 144),
        2
    )

    # =================================================
    # CENTER ROI -> POTHOLES + RAVELLING
    # =================================================

    contours_center, _ = cv2.findContours(
        center_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_center:

        area = cv2.contourArea(cnt)

        if area > min_area:

            x, y, ww, hh = cv2.boundingRect(cnt)

            aspect_ratio = ww / float(hh)

            perimeter = cv2.arcLength(cnt, True)

            if perimeter == 0:
                continue

            circularity = (
                4 * np.pi * area
            ) / (perimeter * perimeter)

            # =================================================
            # POTHOLE DETECTION
            # =================================================

            is_pothole = (
                area > 18000 and
                circularity > 0.08 and
                aspect_ratio < 4.5
            )

            # =================================================
            # POTHOLE
            # =================================================

            if is_pothole:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (0, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "POTHOLE",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2
                )

            # =================================================
            # RAVELLING
            # =================================================

            else:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (255, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "RAVELLING",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 0, 255),
                    2
                )

    # =================================================
    # RIGHT ROI -> EDGE CRACKS
    # =================================================

    contours_right, _ = cv2.findContours(
        right_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_right:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

    cv2.putText(
        output,
        "EDGE CRACKS",
        (right_x1 + 10, right_y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (144, 238, 144),
        2
    )

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        enhanced = enhance_frame(frame)

        edges = build_edges(enhanced)

        output_frame = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        writer.write(output_frame)

        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("Pipeline completed.")
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.
Pipeline completed.
Output saved as: IMG_7555_segmented.mp4


In [16]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    edges = cv2.Canny(blur, 50, 120)

    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# FRAME ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    denoised = gaussian_denoise(gray)

    stretched = contrast_stretch(denoised)

    sharpened = sharpen_image(stretched)

    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# ROI DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.18)
    side_roi_height = int(h * 0.66)

    # Reduced center width
    center_roi_width = int(w * 0.58)
    center_roi_height = int(h * 0.78)

    bottom_margin = 5

    # =================================================
    # LEFT ROI
    # =================================================

    left_x1 = 0
    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)
    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    right_x2 = w
    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASKS
    # =================================================

    left_mask = np.zeros(edges.shape, dtype=np.uint8)
    center_mask = np.zeros(edges.shape, dtype=np.uint8)
    right_mask = np.zeros(edges.shape, dtype=np.uint8)

    cv2.rectangle(
        left_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    cv2.rectangle(
        center_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    cv2.rectangle(
        right_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # =================================================
    # APPLY ROI MASKS
    # =================================================

    left_edges = cv2.bitwise_and(edges, left_mask)
    center_edges = cv2.bitwise_and(edges, center_mask)
    right_edges = cv2.bitwise_and(edges, right_mask)

    # =================================================
    # LEFT ROI -> EDGE CRACKS
    # =================================================

    contours_left, _ = cv2.findContours(
        left_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_left:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

    cv2.putText(
        output,
        "EDGE CRACKS",
        (left_x1 + 10, left_y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (144, 238, 144),
        2
    )

    # =================================================
    # CENTER ROI -> POTHOLES + RAVELLING
    # =================================================

    contours_center, _ = cv2.findContours(
        center_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_center:

        area = cv2.contourArea(cnt)

        if area > min_area:

            x, y, ww, hh = cv2.boundingRect(cnt)

            perimeter = cv2.arcLength(cnt, True)

            if perimeter == 0:
                continue

            circularity = (
                4 * np.pi * area
            ) / (perimeter * perimeter)

            # =================================================
            # POTHOLE DETECTION
            # =================================================

            is_pothole = (
                area > 35000 and
                circularity > 0.03
            )

            # =================================================
            # POTHOLE
            # =================================================

            if is_pothole:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (0, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "POTHOLE",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2
                )

            # =================================================
            # RAVELLING
            # =================================================

            else:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (255, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "RAVELLING",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 0, 255),
                    2
                )

    # =================================================
    # RIGHT ROI -> EDGE CRACKS
    # =================================================

    contours_right, _ = cv2.findContours(
        right_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours_right:

        area = cv2.contourArea(cnt)

        if area > min_area:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

    cv2.putText(
        output,
        "EDGE CRACKS",
        (right_x1 + 10, right_y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (144, 238, 144),
        2
    )

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        enhanced = enhance_frame(frame)

        edges = build_edges(enhanced)

        output_frame = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        writer.write(output_frame)

        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("Pipeline completed.")
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.
Pipeline completed.
Output saved as: IMG_7555_segmented.mp4


In [17]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    edges = cv2.Canny(blur, 50, 120)

    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# FRAME ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    denoised = gaussian_denoise(gray)

    stretched = contrast_stretch(denoised)

    sharpened = sharpen_image(stretched)

    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# DAMAGE CLASSIFICATION
# =====================================================

def classify_and_draw(contours, output, min_area):

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > min_area:

            x, y, w, h = cv2.boundingRect(cnt)

            perimeter = cv2.arcLength(cnt, True)

            if perimeter == 0:
                continue

            circularity = (
                4 * np.pi * area
            ) / (perimeter * perimeter)

            # =================================================
            # POTHOLE CONDITION
            # =================================================
            #
            # Large circular damage -> POTHOLE
            # Small / irregular -> RAVELLING
            #
            # =================================================

            is_pothole = (
                area > 35000 and
                circularity > 0.03
            )

            # =================================================
            # POTHOLE
            # =================================================

            if is_pothole:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (0, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "POTHOLE",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2
                )

            # =================================================
            # RAVELLING
            # =================================================

            else:

                cv2.drawContours(
                    output,
                    [cnt],
                    -1,
                    (255, 0, 255),
                    3
                )

                cv2.putText(
                    output,
                    "RAVELLING",
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 0, 255),
                    2
                )


# =====================================================
# ROI DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.18)
    side_roi_height = int(h * 0.66)

    center_roi_width = int(w * 0.58)
    center_roi_height = int(h * 0.78)

    bottom_margin = 5

    # =================================================
    # LEFT ROI
    # =================================================

    left_x1 = 0
    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)
    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    right_x2 = w
    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASKS
    # =================================================

    left_mask = np.zeros(edges.shape, dtype=np.uint8)
    center_mask = np.zeros(edges.shape, dtype=np.uint8)
    right_mask = np.zeros(edges.shape, dtype=np.uint8)

    cv2.rectangle(
        left_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    cv2.rectangle(
        center_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    cv2.rectangle(
        right_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # =================================================
    # APPLY ROI MASKS
    # =================================================

    left_edges = cv2.bitwise_and(edges, left_mask)
    center_edges = cv2.bitwise_and(edges, center_mask)
    right_edges = cv2.bitwise_and(edges, right_mask)

    # =================================================
    # FIND CONTOURS
    # =================================================

    contours_left, _ = cv2.findContours(
        left_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contours_center, _ = cv2.findContours(
        center_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contours_right, _ = cv2.findContours(
        right_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # =================================================
    # CLASSIFICATION FOR ALL 3 ROIs
    # =================================================

    classify_and_draw(
        contours_left,
        output,
        min_area
    )

    classify_and_draw(
        contours_center,
        output,
        min_area
    )

    classify_and_draw(
        contours_right,
        output,
        min_area
    )

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        enhanced = enhance_frame(frame)

        edges = build_edges(enhanced)

        output_frame = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        writer.write(output_frame)

        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("Pipeline completed.")
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.
Pipeline completed.
Output saved as: IMG_7555_segmented.mp4


In [18]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    edges = cv2.Canny(blur, 50, 120)

    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# FRAME ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    denoised = gaussian_denoise(gray)

    stretched = contrast_stretch(denoised)

    sharpened = sharpen_image(stretched)

    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# DAMAGE CLASSIFICATION
# =====================================================

def classify_damage(cnt, output, region_type, min_area):

    area = cv2.contourArea(cnt)

    if area < min_area:
        return

    x, y, w, h = cv2.boundingRect(cnt)

    perimeter = cv2.arcLength(cnt, True)

    if perimeter == 0:
        return

    circularity = (
        4 * np.pi * area
    ) / (perimeter * perimeter)

    aspect_ratio = w / float(h)

    # =================================================
    # POTHOLE DETECTION
    # =================================================
    #
    # Big compact region -> pothole
    #
    # =================================================

    is_pothole = (
        area > 35000 and
        circularity > 0.03
    )

    # =================================================
    # EDGE CRACK DETECTION
    # =================================================
    #
    # Long thin contours near edges
    #
    # =================================================

    is_edge_crack = (
        aspect_ratio > 4.5 or
        aspect_ratio < 0.22
    )

    # =================================================
    # LEFT + RIGHT REGION
    # =================================================

    if region_type == "SIDE":

        # EDGE CRACKS
        if is_edge_crack:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            cv2.putText(
                output,
                "EDGE CRACK",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

        # POTHOLES
        elif is_pothole:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 0, 255),
                3
            )

            cv2.putText(
                output,
                "POTHOLE",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

        # RAVELLING
        else:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (255, 0, 255),
                3
            )

            cv2.putText(
                output,
                "RAVELLING",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 0, 255),
                2
            )

    # =================================================
    # CENTER REGION
    # =================================================

    else:

        # POTHOLES
        if is_pothole:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 0, 255),
                3
            )

            cv2.putText(
                output,
                "POTHOLE",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2
            )

        # RAVELLING
        else:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (255, 0, 255),
                3
            )

            cv2.putText(
                output,
                "RAVELLING",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 255),
                2
            )


# =====================================================
# ROI DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.18)
    side_roi_height = int(h * 0.66)

    center_roi_width = int(w * 0.58)
    center_roi_height = int(h * 0.78)

    bottom_margin = 5

    # =================================================
    # LEFT ROI
    # =================================================

    left_x1 = 0
    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)
    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    right_x2 = w
    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASKS
    # =================================================

    left_mask = np.zeros(edges.shape, dtype=np.uint8)
    center_mask = np.zeros(edges.shape, dtype=np.uint8)
    right_mask = np.zeros(edges.shape, dtype=np.uint8)

    cv2.rectangle(
        left_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    cv2.rectangle(
        center_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    cv2.rectangle(
        right_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # =================================================
    # APPLY ROI MASKS
    # =================================================

    left_edges = cv2.bitwise_and(edges, left_mask)
    center_edges = cv2.bitwise_and(edges, center_mask)
    right_edges = cv2.bitwise_and(edges, right_mask)

    # =================================================
    # FIND CONTOURS
    # =================================================

    contours_left, _ = cv2.findContours(
        left_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contours_center, _ = cv2.findContours(
        center_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contours_right, _ = cv2.findContours(
        right_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # =================================================
    # LEFT REGION
    # =================================================

    for cnt in contours_left:

        classify_damage(
            cnt,
            output,
            "SIDE",
            min_area
        )

    # =================================================
    # CENTER REGION
    # =================================================

    for cnt in contours_center:

        classify_damage(
            cnt,
            output,
            "CENTER",
            min_area
        )

    # =================================================
    # RIGHT REGION
    # =================================================

    for cnt in contours_right:

        classify_damage(
            cnt,
            output,
            "SIDE",
            min_area
        )

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        enhanced = enhance_frame(frame)

        edges = build_edges(enhanced)

        output_frame = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        writer.write(output_frame)

        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("Pipeline completed.")
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.
Pipeline completed.
Output saved as: IMG_7555_segmented.mp4


In [19]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    edges = cv2.Canny(blur, 50, 120)

    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# FRAME ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    denoised = gaussian_denoise(gray)

    stretched = contrast_stretch(denoised)

    sharpened = sharpen_image(stretched)

    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# DAMAGE CLASSIFICATION
# =====================================================

def classify_damage(cnt, output, region_type, min_area):

    area = cv2.contourArea(cnt)

    if area < min_area:
        return

    x, y, w, h = cv2.boundingRect(cnt)

    perimeter = cv2.arcLength(cnt, True)

    if perimeter == 0:
        return

    circularity = (
        4 * np.pi * area
    ) / (perimeter * perimeter)

    aspect_ratio = w / float(h)

    # =================================================
    # POTHOLE DETECTION
    # =================================================

    is_pothole = (
        area > 35000 and
        circularity > 0.03
    )

    # =================================================
    # EDGE CRACK DETECTION
    # =================================================

    is_edge_crack = (
        aspect_ratio > 4.5 or
        aspect_ratio < 0.22
    )

    # =================================================
    # SIDE REGIONS
    # =================================================

    if region_type == "SIDE":

        # EDGE CRACK
        if is_edge_crack:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            cv2.putText(
                output,
                "EDGE CRACK",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

        # POTHOLE
        elif is_pothole:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 0, 255),
                3
            )

            cv2.putText(
                output,
                "POTHOLE",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

        # RAVELLING
        else:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (255, 0, 255),
                3
            )

            cv2.putText(
                output,
                "RAVELLING",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 0, 255),
                2
            )

    # =================================================
    # CENTER REGION
    # =================================================

    else:

        # POTHOLE
        if is_pothole:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 0, 255),
                3
            )

            cv2.putText(
                output,
                "POTHOLE",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2
            )

        # RAVELLING
        else:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (255, 0, 255),
                3
            )

            cv2.putText(
                output,
                "RAVELLING",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 255),
                2
            )


# =====================================================
# ROI DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.18)
    side_roi_height = int(h * 0.66)

    center_roi_width = int(w * 0.58)
    center_roi_height = int(h * 0.78)

    bottom_margin = 5

    # =================================================
    # LEFT ROI
    # =================================================

    left_x1 = 0
    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)
    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    right_x2 = w
    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASKS
    # =================================================

    left_mask = np.zeros(edges.shape, dtype=np.uint8)
    center_mask = np.zeros(edges.shape, dtype=np.uint8)
    right_mask = np.zeros(edges.shape, dtype=np.uint8)

    cv2.rectangle(
        left_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    cv2.rectangle(
        center_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    cv2.rectangle(
        right_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # =================================================
    # APPLY ROI MASKS
    # =================================================

    left_edges = cv2.bitwise_and(edges, left_mask)
    center_edges = cv2.bitwise_and(edges, center_mask)
    right_edges = cv2.bitwise_and(edges, right_mask)

    # =================================================
    # FIND CONTOURS
    # =================================================

    contours_left, _ = cv2.findContours(
        left_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contours_center, _ = cv2.findContours(
        center_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contours_right, _ = cv2.findContours(
        right_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # =================================================
    # LEFT REGION
    # =================================================

    for cnt in contours_left:

        classify_damage(
            cnt,
            output,
            "SIDE",
            min_area
        )

    # Add left region label
    cv2.putText(
        output,
        "LEFT REGION",
        (left_x1 + 10, left_y1 - 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (144, 238, 144),
        2
    )

    # =================================================
    # CENTER REGION
    # =================================================

    for cnt in contours_center:

        classify_damage(
            cnt,
            output,
            "CENTER",
            min_area
        )

    cv2.putText(
        output,
        "CENTER REGION",
        (center_x1 + 10, center_y1 - 15),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 0),
        2
    )

    # =================================================
    # RIGHT REGION
    # =================================================

    for cnt in contours_right:

        classify_damage(
            cnt,
            output,
            "SIDE",
            min_area
        )

    # Add right region label
    cv2.putText(
        output,
        "RIGHT REGION",
        (right_x1 + 10, right_y1 - 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (144, 238, 144),
        2
    )

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        enhanced = enhance_frame(frame)

        edges = build_edges(enhanced)

        output_frame = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        writer.write(output_frame)

        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("Pipeline completed.")
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()

Starting segmentation pipeline...
Press q to stop preview.
Pipeline completed.
Output saved as: IMG_7555_segmented.mp4


In [ ]:
import cv2
import numpy as np


# =====================================================
# VIDEO SETTINGS
# =====================================================

VIDEO_PATH = "IMG_7555.MOV"
OUTPUT_VIDEO = "IMG_7555_segmented.mp4"

MIN_AREA = 9000


# =====================================================
# DENOISING
# =====================================================

def gaussian_denoise(image):

    smooth = cv2.GaussianBlur(image, (5, 5), 0)

    return smooth


# =====================================================
# CONTRAST STRETCHING
# =====================================================

def contrast_stretch(image):

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return image

    dmin = 50
    dmax = 200

    stretched = dmin + (image - min_val) * (
        (dmax - dmin) / (max_val - min_val)
    )

    return stretched.astype("uint8")


# =====================================================
# SHARPENING
# =====================================================

def sharpen_image(image):

    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])

    sharp = cv2.filter2D(image, -1, kernel)

    return sharp


# =====================================================
# EDGE BUILDING
# =====================================================

def build_edges(frame):

    if frame is None:
        raise ValueError("Input frame is None.")

    if len(frame.shape) == 2:
        gray = frame
    else:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.bilateralFilter(gray, 9, 75, 75)

    edges = cv2.Canny(blur, 50, 120)

    kernel = np.ones((5, 5), np.uint8)

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=3
    )

    return edges


# =====================================================
# FRAME ENHANCEMENT
# =====================================================

def enhance_frame(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    denoised = gaussian_denoise(gray)

    stretched = contrast_stretch(denoised)

    sharpened = sharpen_image(stretched)

    enhanced = cv2.cvtColor(
        sharpened,
        cv2.COLOR_GRAY2BGR
    )

    return enhanced


# =====================================================
# DAMAGE CLASSIFICATION
# =====================================================

def classify_damage(cnt, output, region_type, min_area):

    area = cv2.contourArea(cnt)

    if area < min_area:
        return

    x, y, w, h = cv2.boundingRect(cnt)

    perimeter = cv2.arcLength(cnt, True)

    if perimeter == 0:
        return

    circularity = (
        4 * np.pi * area
    ) / (perimeter * perimeter)

    aspect_ratio = w / float(h)

    # =================================================
    # SIDE REGIONS
    # =================================================

    if region_type == "SIDE":

        # =================================================
        # EDGE CRACK (HIGH PRIORITY)
        # =================================================

        if (

            aspect_ratio > 1.8

            or

            aspect_ratio < 0.55

            or

            (
                area < 40000 and
                circularity < 0.08
            )
        ):

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 255, 0),
                3
            )

            cv2.putText(
                output,
                "EDGE CRACK",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

        # =================================================
        # POTHOLE
        # =================================================

        elif (
            area > 35000 and
            circularity > 0.03
        ):

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 0, 255),
                3
            )

            cv2.putText(
                output,
                "POTHOLE",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

        # =================================================
        # RAVELLING
        # =================================================

        else:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (255, 0, 255),
                3
            )

            cv2.putText(
                output,
                "RAVELLING",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 0, 255),
                2
            )

    # =================================================
    # CENTER REGION
    # =================================================

    else:

        # POTHOLE

        if (
            area > 35000 and
            circularity > 0.03
        ):

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (0, 0, 255),
                3
            )

            cv2.putText(
                output,
                "POTHOLE",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2
            )

        # RAVELLING

        else:

            cv2.drawContours(
                output,
                [cnt],
                -1,
                (255, 0, 255),
                3
            )

            cv2.putText(
                output,
                "RAVELLING",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 255),
                2
            )


# =====================================================
# ROI DETECTION
# =====================================================

def draw_large_contours_roi(frame, edges, min_area=9000):

    output = frame.copy()

    h, w = frame.shape[:2]

    # =================================================
    # ROI SETTINGS
    # =================================================

    side_roi_width = int(w * 0.18)
    side_roi_height = int(h * 0.66)

    center_roi_width = int(w * 0.58)
    center_roi_height = int(h * 0.78)

    bottom_margin = 5

    # =================================================
    # LEFT ROI
    # =================================================

    left_x1 = 0
    left_y1 = h - side_roi_height - bottom_margin

    left_x2 = left_x1 + side_roi_width
    left_y2 = h - bottom_margin

    # =================================================
    # CENTER ROI
    # =================================================

    center_x1 = (w // 2) - (center_roi_width // 2)
    center_y1 = h - center_roi_height - bottom_margin

    center_x2 = center_x1 + center_roi_width
    center_y2 = h - bottom_margin

    # =================================================
    # RIGHT ROI
    # =================================================

    right_x2 = w
    right_x1 = right_x2 - side_roi_width

    right_y1 = h - side_roi_height - bottom_margin
    right_y2 = h - bottom_margin

    # =================================================
    # CREATE ROI MASKS
    # =================================================

    left_mask = np.zeros(edges.shape, dtype=np.uint8)
    center_mask = np.zeros(edges.shape, dtype=np.uint8)
    right_mask = np.zeros(edges.shape, dtype=np.uint8)

    cv2.rectangle(
        left_mask,
        (left_x1, left_y1),
        (left_x2, left_y2),
        255,
        -1
    )

    cv2.rectangle(
        center_mask,
        (center_x1, center_y1),
        (center_x2, center_y2),
        255,
        -1
    )

    cv2.rectangle(
        right_mask,
        (right_x1, right_y1),
        (right_x2, right_y2),
        255,
        -1
    )

    # =================================================
    # APPLY ROI MASKS
    # =================================================

    left_edges = cv2.bitwise_and(edges, left_mask)
    center_edges = cv2.bitwise_and(edges, center_mask)
    right_edges = cv2.bitwise_and(edges, right_mask)

    # =================================================
    # FIND CONTOURS
    # =================================================

    contours_left, _ = cv2.findContours(
        left_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contours_center, _ = cv2.findContours(
        center_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contours_right, _ = cv2.findContours(
        right_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # =================================================
    # LEFT REGION
    # =================================================

    for cnt in contours_left:

        classify_damage(
            cnt,
            output,
            "SIDE",
            min_area
        )

    # =================================================
    # CENTER REGION
    # =================================================

    for cnt in contours_center:

        classify_damage(
            cnt,
            output,
            "CENTER",
            min_area
        )

    # =================================================
    # RIGHT REGION
    # =================================================

    for cnt in contours_right:

        classify_damage(
            cnt,
            output,
            "SIDE",
            min_area
        )

    # =================================================
    # DRAW ROI BOXES
    # =================================================

    cv2.rectangle(
        output,
        (left_x1, left_y1),
        (left_x2, left_y2),
        (144, 238, 144),
        3
    )

    cv2.rectangle(
        output,
        (center_x1, center_y1),
        (center_x2, center_y2),
        (255, 255, 0),
        3
    )

    cv2.rectangle(
        output,
        (right_x1, right_y1),
        (right_x2, right_y2),
        (144, 238, 144),
        3
    )

    return output


# =====================================================
# MAIN VIDEO PIPELINE
# =====================================================

def process_video():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        fourcc,
        fps,
        (width, height)
    )

    print("Starting segmentation pipeline...")
    print("Press q to stop preview.")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        enhanced = enhance_frame(frame)

        edges = build_edges(enhanced)

        output_frame = draw_large_contours_roi(
            enhanced,
            edges,
            min_area=MIN_AREA
        )

        writer.write(output_frame)

        cv2.imshow(
            "Road Damage Segmentation",
            output_frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    writer.release()

    cv2.destroyAllWindows()

    print("Pipeline completed.")
    print("Output saved as:", OUTPUT_VIDEO)


# =====================================================
# RUN PROGRAM
# =====================================================

if __name__ == "__main__":

    process_video()